# Bluesky Dataset Cleaning Pipeline
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

Cleans the Bluesky post CSV files (one per asset class) collected via the authenticated AT Protocol search endpoint. Each file is processed and saved independently, preserving the per-asset structure needed for separate sentiment analysis.

**Workflow:**
1. Upload all per-asset Bluesky CSVs at once.
2. Run cleaning on each file.
3. Download all cleaned files as a ZIP.

**Coverage note:**
Bluesky opened to the public in early 2023 but saw major growth after February 2024 when it lifted the invite-only restriction. The cleaning pipeline still enforces the full study period (2020 to 2024) but in practice almost all posts will fall within March 2023 to December 2024.

**Cleaning steps applied:**
- Remove posts with missing or empty text.
- Enforce the dissertation study period.
- Apply Bluesky-tailored length thresholds (posts are short-form, similar to Twitter).
- Filter out spam patterns (giveaways, airdrops, referral codes, follow-for-follow).
- Remove bot-generated content (price ticker bots, crossposting bots, repetitive link drops).
- Language filter (English only, preserving the consistency of the sentiment classifier).
- Asset relevance check using per-asset keyword lists.
- Deduplicate by post URI if present and by exact text match otherwise.

## 0. Setup & Dependencies

In [ ]:
!pip install -q pandas langdetect

import pandas as pd
import re
import os
import shutil
from datetime import datetime
from pathlib import Path
from google.colab import files

try:
    from langdetect import detect, LangDetectException
    HAS_LANGDETECT = True
except ImportError:
    HAS_LANGDETECT = False

os.makedirs('data/raw/bluesky', exist_ok=True)
os.makedirs('data/processed/bluesky', exist_ok=True)
os.makedirs('data/processed/bluesky/stats', exist_ok=True)
print('Setup complete.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Setup complete.


## 1. Configuration

Bluesky posts are structurally similar to tweets (short-form, 300 character limit) but the platform has slightly different spam patterns. The config below is tuned for Bluesky's characteristics rather than reused from the Twitter script unchanged.

In [ ]:
# --- Study period ---
START_DATE = pd.Timestamp('2020-01-01')
END_DATE = pd.Timestamp('2024-12-31')

# --- Length thresholds ---
# Bluesky has a 300-character limit per post, similar to Twitter's 280.
MIN_TEXT_LENGTH = 15
MAX_TEXT_LENGTH = 300

# --- Spam keywords ---
# Patterns commonly found in Bluesky crypto spam. Some overlap with
# Twitter spam, plus some Bluesky-specific ones (shameless plugs for
# discord invites, starter packs, custom feeds).
SPAM_KEYWORDS = [
    'airdrop', 'free crypto', 'free bitcoin', 'giveaway',
    'claim now', 'click here', 'click my bio', 'link in bio',
    'pump signal', '100x gem', '1000x potential',
    'next moonshot', 'moonshot alert',
    'join my discord', 'join our discord', 'join my telegram',
    'telegram group', 'dm for signals', 'dm me for',
    'premium signals', 'vip signals', 'vip group', 'join vip',
    'referral code', 'referral link', 'use my code',
    'sign up bonus', 'limited time offer',
    'guaranteed profit', 'guaranteed returns',
    'private key', 'wallet hack', 'send me 1 btc', 'send eth',
    'follow back', 'follow4follow', 'f4f',
    'shill my', 'shilling my',
    'check my starter pack',  # Bluesky-specific
    'subscribe to my feed',   # Bluesky-specific
]

# --- Bot patterns ---
BOT_PATTERNS = [
    r'^[A-Z]{3,5}\s*[:|]\s*\$[\d,\.]+',       # Price ticker bots
    r'^(\s*#\w+\s*){4,}$',                    # Pure hashtag spam
    r'^\s*https?://\S+\s*$',                  # Link-only posts
    r'^(\s*@[\w\.-]+\s*){2,}$',              # Mention-only posts
    r'(.)\1{6,}',                              # Repeated character spam
    r'crossposted (from|via)',                 # Crossposter bots
    r'via @skyfeed',                           # Auto-crosspost marker
]

# --- Asset-specific relevance keywords ---
ASSET_KEYWORDS = {
    'bitcoin':        ['bitcoin', 'btc', 'satoshi'],
    'ethereum':       ['ethereum', 'ether', 'eth', 'vitalik'],
    'dogecoin':       ['dogecoin', 'doge'],
    'shiba_inu':      ['shiba inu', 'shib', 'shibarmy', 'shiba'],
    'tether':         ['tether', 'usdt', 'stablecoin'],
    'crypto_general': ['crypto', 'cryptocurrency', 'bitcoin',
                       'ethereum', 'blockchain', 'defi', 'altcoin'],
}

print(f'Study period: {START_DATE.date()} to {END_DATE.date()}')
print(f'Asset categories: {list(ASSET_KEYWORDS.keys())}')

Study period: 2020-01-01 to 2024-12-31
Asset categories: ['bitcoin', 'ethereum', 'dogecoin', 'shiba_inu', 'tether', 'crypto_general']


In [ ]:
# --- Helper functions ---

def clean_text(text):
    if not isinstance(text, str):
        return ''
    # Remove URLs for filtering purposes (they will be kept in the final output).
    # We use a copy for pattern detection only, so we do not mutate the text here.
    text = text.replace('\xa0', ' ')
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_english(text, language_col_value=None):
    if language_col_value is not None and pd.notna(language_col_value):
        lang_str = str(language_col_value).lower().strip()
        return lang_str in ('en', 'english', 'en-us', 'en-gb', '')
    if HAS_LANGDETECT and text and len(text) > 10:
        try:
            return detect(text) == 'en'
        except LangDetectException:
            return False
    return True

def is_spam(text):
    if not text:
        return True
    text_lower = text.lower()
    return any(kw in text_lower for kw in SPAM_KEYWORDS)

def is_bot_pattern(text):
    if not text:
        return True
    return any(re.search(p, text, re.IGNORECASE) for p in BOT_PATTERNS)

def is_relevant(text, keywords):
    if not text or not keywords:
        return False
    text_lower = text.lower()
    return any(kw in text_lower for kw in keywords)

def detect_asset_from_filename(filename):
    fname_lower = filename.lower()
    for asset in ['shiba_inu', 'shibainu', 'shibarmy', 'shib',
                  'crypto_general', 'crypto general', 'general',
                  'bitcoin', 'ethereum', 'dogecoin', 'tether']:
        if asset in fname_lower:
            if asset in ('shibainu', 'shibarmy', 'shib'):
                return 'shiba_inu'
            if asset in ('crypto general', 'general'):
                return 'crypto_general'
            return asset
    return None

def detect_columns(df):
    cols = {col.lower(): col for col in df.columns}
    text_col = next((cols[c] for c in ['text','content','post_text','record_text']
                     if c in cols), None)
    date_col = next((cols[c] for c in ['created_at','date','timestamp',
                                         'indexed_at','createdat']
                     if c in cols), None)
    uri_col = next((cols[c] for c in ['uri','post_uri','id','cid']
                    if c in cols), None)
    lang_col = next((cols[c] for c in ['language','lang','langs']
                     if c in cols), None)
    author_col = next((cols[c] for c in ['author','author_handle','handle',
                                           'username','user']
                       if c in cols), None)
    return text_col, date_col, uri_col, lang_col, author_col

print('Helper functions loaded.')

Helper functions loaded.


## 2. Upload All Bluesky CSVs at Once

Select every per-asset Bluesky CSV. Hold Shift or Cmd/Ctrl to multi-select. Files are saved to `data/raw/bluesky/`.

In [ ]:
print('Upload all your Bluesky dataset CSVs (one per asset class):\n')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/raw/bluesky/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content)/1e6:.2f} MB)')

print(f'\n{len(uploaded)} file(s) uploaded.')

Upload all your Bluesky dataset CSVs (one per asset class):



Saving bitcoin_bluesky.csv to bitcoin_bluesky.csv
  Saved: data/raw/bluesky/bitcoin_bluesky.csv (18.29 MB)

1 file(s) uploaded.


## 3. Cleaning Function

Applies all filters sequentially: missing text, date range, length, language, spam, bot patterns, asset relevance, and deduplication by URI and exact text match.

In [ ]:
def clean_bluesky_file(input_path, output_path, stats_path, asset_keywords):
    print('\n' + '=' * 60)
    print(f'CLEANING: {os.path.basename(input_path)}')
    print('=' * 60)

    try:
        df = pd.read_csv(input_path, on_bad_lines='skip',
                         engine='python', encoding='utf-8',
                         encoding_errors='ignore')
    except Exception as e:
        print(f'  ERROR loading file: {e}')
        return None

    initial = len(df)
    print(f'  Loaded: {initial:,} posts')

    text_col, date_col, uri_col, lang_col, author_col = detect_columns(df)
    print(f'  Text: {text_col} | Date: {date_col} | URI: {uri_col} | '
          f'Lang: {lang_col} | Author: {author_col}')

    if not text_col:
        print(f'  ERROR: No text column. Cols: {list(df.columns)}')
        return None

    # Normalise the text column.
    df[text_col] = df[text_col].fillna('').astype(str).apply(clean_text)

    stats = {k: 0 for k in ['initial', 'dropped_no_text', 'dropped_date',
                             'dropped_length', 'dropped_language',
                             'dropped_spam', 'dropped_bot',
                             'dropped_irrelevant', 'dropped_duplicate', 'kept']}
    stats['initial'] = initial

    # 1. Drop empty text.
    b = len(df); df = df[df[text_col].str.len() > 0]
    stats['dropped_no_text'] = b - len(df)

    # 2. Date range.
    if date_col:
        try:
            parsed = pd.to_datetime(df[date_col], errors='coerce', utc=True)
            df['_parsed_date'] = parsed.dt.tz_localize(None)
            b = len(df)
            df = df[(df['_parsed_date'] >= START_DATE) &
                    (df['_parsed_date'] <= END_DATE)]
            stats['dropped_date'] = b - len(df)
        except Exception as e:
            print(f'  Warning: date filter failed ({e})')

    # 3. Length filter.
    b = len(df); lens = df[text_col].str.len()
    df = df[(lens >= MIN_TEXT_LENGTH) & (lens <= MAX_TEXT_LENGTH)]
    stats['dropped_length'] = b - len(df)

    # 4. Language.
    if lang_col and lang_col in df.columns:
        b = len(df)
        df = df[df[lang_col].apply(lambda x: is_english(None, x))]
        stats['dropped_language'] = b - len(df)
    elif HAS_LANGDETECT:
        b = len(df); df = df[df[text_col].apply(is_english)]
        stats['dropped_language'] = b - len(df)

    # 5. Spam.
    b = len(df); df = df[~df[text_col].apply(is_spam)]
    stats['dropped_spam'] = b - len(df)

    # 6. Bot patterns.
    b = len(df); df = df[~df[text_col].apply(is_bot_pattern)]
    stats['dropped_bot'] = b - len(df)

    # 7. Relevance.
    b = len(df)
    df = df[df[text_col].apply(lambda t: is_relevant(t, asset_keywords))]
    stats['dropped_irrelevant'] = b - len(df)

    # 8. Deduplicate.
    b = len(df)
    if uri_col and uri_col in df.columns:
        df = df.drop_duplicates(subset=[uri_col], keep='first')
    df = df.drop_duplicates(subset=[text_col], keep='first')
    stats['dropped_duplicate'] = b - len(df)

    # Standardise the date column if present.
    if '_parsed_date' in df.columns:
        df['date'] = df['_parsed_date'].dt.strftime('%Y-%m-%d')
        df = df.drop(columns=['_parsed_date'])

    stats['kept'] = len(df)
    df.to_csv(output_path, index=False)

    print(f'\n  --- Summary ---')
    for key, val in stats.items():
        print(f'  {key:25s} {val:>10,}')
    if initial > 0:
        print(f'  {"retention":25s} {100 * stats["kept"] / initial:>9.1f}%')
    if 'date' in df.columns and len(df) > 0:
        print(f'  Date range: {df["date"].min()} to {df["date"].max()}')
    print(f'  Saved: {output_path}')

    if stats_path:
        with open(stats_path, 'w') as f:
            f.write(f'Bluesky Cleaning Statistics\nFile: {input_path}\n')
            f.write(f'Run: {datetime.now().isoformat()}\n' + '=' * 50 + '\n\n')
            for key, val in stats.items():
                f.write(f'{key}: {val:,}\n')

    return stats

print('Cleaning function loaded.')

Cleaning function loaded.


## 4. Process All Uploaded Files

Iterates through every uploaded Bluesky CSV, detects the target asset from each filename, and runs the cleaning pipeline with the appropriate asset keywords.

In [ ]:
INPUT_DIR = 'data/raw/bluesky'
OUTPUT_DIR = 'data/processed/bluesky'
STATS_DIR = 'data/processed/bluesky/stats'

csv_files = sorted([f for f in os.listdir(INPUT_DIR)
                    if f.endswith('.csv') and not f.startswith('all_')])

if not csv_files:
    print(f'No CSV files found in {INPUT_DIR}. Upload files in Step 2 first.')
else:
    print('=' * 60)
    print(f'PROCESSING {len(csv_files)} BLUESKY FILE(S)')
    print('=' * 60)
    for f in csv_files:
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, f)) / 1e6
        print(f'  {f} ({size_mb:.2f} MB)')

    overall_stats = {}
    for fname in csv_files:
        input_path = os.path.join(INPUT_DIR, fname)
        output_path = os.path.join(OUTPUT_DIR,
                                    fname.replace('.csv', '_cleaned.csv'))
        stats_path = os.path.join(STATS_DIR,
                                   fname.replace('.csv', '_stats.txt'))

        asset = detect_asset_from_filename(fname)
        if not asset:
            print(f'\n  Note: could not infer asset for {fname}, '
                  f'using general crypto keywords.')
            asset = 'crypto_general'
        keywords = ASSET_KEYWORDS.get(asset, ASSET_KEYWORDS['crypto_general'])
        print(f'\n  Detected asset: {asset}')
        print(f'  Keywords: {keywords}')

        stats = clean_bluesky_file(input_path, output_path,
                                    stats_path, keywords)
        if stats:
            overall_stats[fname] = stats

    # Final summary.
    print('\n' + '=' * 60)
    print('OVERALL SUMMARY')
    print('=' * 60)
    total_initial = sum(s['initial'] for s in overall_stats.values())
    total_kept = sum(s['kept'] for s in overall_stats.values())
    print(f'Files processed:   {len(overall_stats)}')
    print(f'Total input posts: {total_initial:,}')
    print(f'Total kept posts:  {total_kept:,}')
    if total_initial > 0:
        print(f'Overall retention: {100 * total_kept / total_initial:.1f}%')

    print(f'\nPer-file results:')
    for fname, stats in overall_stats.items():
        ret = 100 * stats['kept'] / stats['initial'] if stats['initial'] else 0
        print(f'  {fname:50s} {stats["kept"]:>6,} kept ({ret:5.1f}%)')

PROCESSING 1 BLUESKY FILE(S)
  bitcoin_bluesky.csv (18.29 MB)

  Detected asset: bitcoin
  Keywords: ['bitcoin', 'btc', 'satoshi']

CLEANING: bitcoin_bluesky.csv
  Loaded: 51,654 posts
  Text: text | Date: created_at | URI: id | Lang: language | Author: author_handle

  --- Summary ---
  initial                       51,654
  dropped_no_text                  310
  dropped_date                   6,714
  dropped_length                   988
  dropped_language                   0
  dropped_spam                     212
  dropped_bot                      422
  dropped_irrelevant             2,436
  dropped_duplicate              1,459
  kept                          39,113
  retention                      75.7%
  Date range: 2023-03-01 to 2024-12-30
  Saved: data/processed/bluesky/bitcoin_bluesky_cleaned.csv

OVERALL SUMMARY
Files processed:   1
Total input posts: 51,654
Total kept posts:  39,113
Overall retention: 75.7%

Per-file results:
  bitcoin_bluesky.csv                              

## 5. Download All Cleaned Files

In [ ]:
zip_name = 'bluesky_cleaned'
shutil.make_archive(zip_name, 'zip', 'data/processed', 'bluesky')
zip_path = f'{zip_name}.zip'
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Archive: {zip_path} ({size_mb:.2f} MB)')
files.download(zip_path)
print('Download started.')

Archive: bluesky_cleaned.zip (3.69 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
